# Notebook A v3 — Trace Generator Mix

Purpose: generate a safer SFT trace corpus for the NVIDIA Nemotron Model Reasoning Challenge.

This v3 mix is driven by Notebook D validation:
- **P0 bit manipulation repair**: short traces, no repeated "Ignore story text", always one boxed 8-bit final answer.
- **P1 cipher support**: concise symbolic/string trace, always boxed.
- **P2 clean equation numeric subset**: conservative short traces only.
- **Preserve numeral/unit/gravity** as calibration / anti-forgetting examples.
- **Defer cryptarithm** from training by default.

Output:
- `/kaggle/working/train_traces_v3_mix.jsonl`
- `/kaggle/working/train_traces_v3_mix_summary.csv`
- `/kaggle/working/train_traces_v3_mix.zip`


In [1]:
from pathlib import Path
import json
import random
import re
import zipfile
from collections import Counter

import pandas as pd

RANDOM_SEED = 42
OUTPUT_JSONL_NAME = "train_traces_v3_mix.jsonl"
OUTPUT_ZIP_NAME = "train_traces_v3_mix.zip"
OUTPUT_SUMMARY_NAME = "train_traces_v3_mix_summary.csv"

# Conservative first-pass mix. We intentionally do not oversample categories already strong in Notebook D.
SAMPLE_PLAN = {
    "bit_manipulation": 256,          # P0 repair
    "cipher": 192,                    # P1 new support
    "equation_numeric_deduce": 96,    # P2 clean/short subset
    "gravity": 64,                    # preserve
    "unit_conversion": 64,            # preserve
    "numeral": 64,                    # preserve
    # cryptarithm_deduce intentionally excluded by default
}

MAX_ASSISTANT_CHARS = 900
FINAL_INSTRUCTION = "Please put your final answer inside `\\boxed{}`."

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print("RANDOM_SEED:", RANDOM_SEED)
print("SAMPLE_PLAN:", SAMPLE_PLAN)
print("OUTPUT_JSONL_NAME:", OUTPUT_JSONL_NAME)


RANDOM_SEED: 42
SAMPLE_PLAN: {'bit_manipulation': 256, 'cipher': 192, 'equation_numeric_deduce': 96, 'gravity': 64, 'unit_conversion': 64, 'numeral': 64}
OUTPUT_JSONL_NAME: train_traces_v3_mix.jsonl


In [2]:
def find_train_csv() -> Path:
    candidates = [
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
        Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"),
        Path("/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = sorted(Path("/kaggle/input").glob("**/train.csv"))
    if not matches:
        raise FileNotFoundError("Could not find train.csv under /kaggle/input")
    return matches[0]

TRAIN_CSV = find_train_csv()
train_df = pd.read_csv(TRAIN_CSV)
print("TRAIN_CSV:", TRAIN_CSV)
print("train shape:", train_df.shape)
print("columns:", list(train_df.columns))

id_col = "id" if "id" in train_df.columns else None
prompt_col = "prompt" if "prompt" in train_df.columns else "problem"
answer_col = "answer"

assert prompt_col in train_df.columns
assert answer_col in train_df.columns


TRAIN_CSV: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
train shape: (9500, 3)
columns: ['id', 'prompt', 'answer']


In [3]:
QUERY_OP_RE = re.compile(r'(?:query|question|find|solve).*?([+\-*/^=<>]+|[A-Za-z_][A-Za-z0-9_]*)', re.IGNORECASE | re.DOTALL)

def _examples_text(prompt: str) -> str:
    lower = prompt.lower()
    cut_points = [lower.find(marker) for marker in ["query", "question", "now solve", "find the", "determine the result"] if lower.find(marker) >= 0]
    if not cut_points:
        return prompt
    return prompt[:min(cut_points)]

def _query_operator(prompt: str):
    m = QUERY_OP_RE.search(prompt)
    if m:
        return m.group(1)
    tail = prompt[-800:]
    ops = re.findall(r'([+\-*/^=<>]+)', tail)
    return ops[-1] if ops else None

def _operator_appears_in_examples(prompt: str) -> bool:
    query_op = _query_operator(prompt)
    if not query_op:
        return False
    return query_op in _examples_text(prompt)

def detect_category(prompt):
    text = str(prompt)
    lower = text.lower()
    if "secret bit manipulation rule transforms 8-bit binary numbers" in lower:
        return "bit_manipulation"
    if "secret encryption rules are used on text" in lower:
        return "cipher"
    if "secret set of transformation rules is applied to equations" in lower:
        examples = _examples_text(text)
        has_digits = bool(re.search(r"\d", examples))
        op_seen = _operator_appears_in_examples(text)
        if has_digits:
            return "equation_numeric_deduce" if op_seen else "equation_numeric_guess"
        return "cryptarithm_deduce" if op_seen else "cryptarithm_guess"
    if "gravitational constant has been secretly changed" in lower:
        return "gravity"
    if "converted into a different numeral system" in lower:
        return "numeral"
    if "secret unit conversion is applied to measurements" in lower:
        return "unit_conversion"
    return "unknown"

train_df["category"] = train_df[prompt_col].map(detect_category)
print(train_df["category"].value_counts(dropna=False).to_string())


category
bit_manipulation           1602
gravity                    1597
unit_conversion            1594
numeral                    1576
cipher                     1576
equation_numeric_deduce     654
cryptarithm_deduce          554
cryptarithm_guess           269
equation_numeric_guess       78


In [4]:
def boxed(answer: str) -> str:
    return "\\boxed{" + str(answer).strip() + "}"

def clean_answer(answer) -> str:
    return str(answer).strip()

def infer_query_line(prompt: str) -> str:
    lines = [ln.strip() for ln in str(prompt).splitlines() if ln.strip()]
    for ln in reversed(lines):
        if any(k in ln.lower() for k in ["determine", "result", "convert", "what", "find", "now"]):
            return ln[:220]
    return lines[-1][:220] if lines else ""

def make_user(prompt: str) -> str:
    return str(prompt).rstrip() + "\n" + FINAL_INSTRUCTION

def assistant_bit(prompt: str, answer: str) -> str:
    ans = clean_answer(answer)
    # Preserve leading zeros and enforce exact 8-bit answer.
    if not re.fullmatch(r"[01]{8}", ans):
        raise ValueError(f"bit answer is not exactly 8 bits: {ans!r}")
    query = infer_query_line(prompt)
    return (
        "Task: infer an 8-bit bit-transformation rule from the examples. "
        "Keep leading zeros and stop after the final answer.\n"
        f"Query: {query}\n"
        f"Output bits: {ans}\n"
        f"Final answer: {boxed(ans)}"
    )

def assistant_cipher(prompt: str, answer: str) -> str:
    ans = clean_answer(answer)
    query = infer_query_line(prompt)
    return (
        "Task: infer the demonstrated string transformation and apply it once to the query.\n"
        f"Query: {query}\n"
        f"Transformed string: {ans}\n"
        f"Final answer: {boxed(ans)}"
    )

def assistant_equation_numeric(prompt: str, answer: str) -> str:
    ans = clean_answer(answer)
    query = infer_query_line(prompt)
    return (
        "Task: infer the hidden numeric rule from the worked equations and apply it to the query.\n"
        f"Query: {query}\n"
        f"Computed value: {ans}\n"
        f"Final answer: {boxed(ans)}"
    )

def assistant_easy(prompt: str, answer: str, category: str) -> str:
    ans = clean_answer(answer)
    if category == "gravity":
        lead = "Use the altered-gravity examples to infer the relationship, then compute the requested value."
    elif category == "unit_conversion":
        lead = "Use the examples to infer the secret unit conversion, then convert the requested measurement."
    elif category == "numeral":
        lead = "Use the examples to infer the numeral system, then convert the requested value."
    else:
        lead = "Use the examples to infer the rule, then answer."
    query = infer_query_line(prompt)
    return f"{lead}\nQuery: {query}\nResult: {ans}\nFinal answer: {boxed(ans)}"

def build_record(row):
    cat = row["category"]
    prompt = str(row[prompt_col])
    answer = clean_answer(row[answer_col])
    if cat == "bit_manipulation":
        assistant = assistant_bit(prompt, answer)
    elif cat == "cipher":
        assistant = assistant_cipher(prompt, answer)
    elif cat == "equation_numeric_deduce":
        assistant = assistant_equation_numeric(prompt, answer)
    elif cat in {"gravity", "unit_conversion", "numeral"}:
        assistant = assistant_easy(prompt, answer, cat)
    else:
        return None

    if len(assistant) > MAX_ASSISTANT_CHARS:
        assistant = assistant[:MAX_ASSISTANT_CHARS].rstrip() + "\nFinal answer: " + boxed(answer)

    rid = row[id_col] if id_col else str(row.name)
    return {
        "id": str(rid),
        "category": cat,
        "answer": answer,
        "trace_version": "v3_mix_20260525",
        "messages": [
            {"role": "user", "content": make_user(prompt)},
            {"role": "assistant", "content": assistant},
        ],
        "approx_trace_chars": len(assistant),
    }


In [5]:
selected_records = []
selection_counts = Counter()
skipped = []

for category, n in SAMPLE_PLAN.items():
    pool = train_df[train_df["category"] == category].copy()
    if pool.empty:
        print("WARN: no pool for", category)
        continue

    # P2: equation_numeric clean subset — keep short prompts first to reduce noisy long examples.
    if category == "equation_numeric_deduce":
        pool["_prompt_len"] = pool[prompt_col].astype(str).str.len()
        pool = pool.sort_values("_prompt_len").head(max(n * 3, n))

    rng = random.Random(RANDOM_SEED + abs(hash(category)) % 100000)
    idxs = list(pool.index)
    rng.shuffle(idxs)

    take_records = []
    for idx in idxs:
        row = train_df.loc[idx]
        try:
            rec = build_record(row)
        except Exception as exc:
            skipped.append({"id": str(row[id_col] if id_col else idx), "category": category, "reason": repr(exc)})
            continue
        if rec is not None:
            take_records.append(rec)
        if len(take_records) >= n:
            break

    selected_records.extend(take_records)
    selection_counts[category] = len(take_records)

random.Random(RANDOM_SEED).shuffle(selected_records)

print("Selected records:", len(selected_records))
print("Selection counts:", selection_counts)
print("Skipped:", len(skipped))
if skipped[:5]:
    print("First skipped:", skipped[:5])


Selected records: 736
Selection counts: Counter({'bit_manipulation': 256, 'cipher': 192, 'equation_numeric_deduce': 96, 'gravity': 64, 'unit_conversion': 64, 'numeral': 64})
Skipped: 0


In [6]:
def qa_record(rec):
    assistant = rec["messages"][-1]["content"]
    cat = rec["category"]
    ans = rec["answer"]

    if "\\boxed{" not in assistant:
        return False, "missing_boxed"
    if assistant.count("\\boxed{") != 1:
        return False, "multiple_boxed"
    if "Ignore story text" in assistant:
        return False, "forbidden_repetition_phrase"
    if len(assistant) > MAX_ASSISTANT_CHARS + 80:
        return False, "assistant_too_long"

    if cat == "bit_manipulation":
        if not re.fullmatch(r"[01]{8}", ans):
            return False, "bad_bit_answer"

        expected_box = f"\\boxed{{{ans}}}"
        if expected_box not in assistant:
            return False, "bit_box_not_8_binary"

    return True, "ok"

qa_rows = []
good_records = []
for rec in selected_records:
    ok, reason = qa_record(rec)
    qa_rows.append({
        "id": rec["id"],
        "category": rec["category"],
        "ok": ok,
        "reason": reason,
        "chars": rec["approx_trace_chars"],
    })
    if ok:
        good_records.append(rec)

qa_df = pd.DataFrame(qa_rows)
print("QA summary:")
print(qa_df.groupby(["category", "reason"]).size().reset_index(name="count").to_string(index=False))

assert len(good_records) == len(selected_records), "Some generated records failed QA; inspect qa_df"
assert all("Ignore story text" not in r["messages"][-1]["content"] for r in good_records)
print("QA passed:", len(good_records))

QA summary:
               category reason  count
       bit_manipulation     ok    256
                 cipher     ok    192
equation_numeric_deduce     ok     96
                gravity     ok     64
                numeral     ok     64
        unit_conversion     ok     64
QA passed: 736


In [7]:
jsonl_path = WORKING_DIR / OUTPUT_JSONL_NAME
summary_path = WORKING_DIR / OUTPUT_SUMMARY_NAME
zip_path = WORKING_DIR / OUTPUT_ZIP_NAME

with open(jsonl_path, "w", encoding="utf-8") as f:
    for rec in good_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

counts = Counter(r["category"] for r in good_records)
summary_df = pd.DataFrame([{"category": k, "count": v} for k, v in counts.items()]).sort_values("category")
summary_df.to_csv(summary_path, index=False)

if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(jsonl_path, arcname=OUTPUT_JSONL_NAME)
    zf.write(summary_path, arcname=OUTPUT_SUMMARY_NAME)

print("Wrote:", jsonl_path, jsonl_path.stat().st_size)
print("Wrote:", summary_path, summary_path.stat().st_size)
print("Wrote:", zip_path, f"{zip_path.stat().st_size/1024:.1f} KB")
print(summary_df.to_string(index=False))


Wrote: /kaggle/working/train_traces_v3_mix.jsonl 624047
Wrote: /kaggle/working/train_traces_v3_mix_summary.csv 115
Wrote: /kaggle/working/train_traces_v3_mix.zip 73.9 KB
               category  count
       bit_manipulation    256
                 cipher    192
equation_numeric_deduce     96
                gravity     64
                numeral     64
        unit_conversion     64


## Next Notebook B settings

Use this new trace file as Notebook B input:

```python
TRACE_JSONL_NAME = "train_traces_v3_mix.jsonl"
OUTPUT_TAG = "adapter_sft_v3_mix_bitfix_cipher_eq96"
USE_ASSISTANT_ONLY_LOSS = True
MAX_SEQ_LEN = 384  # or 512 if memory permits
```

Do not select any new leaderboard submission unless it beats current best **0.56**.
